# Syndrome affinity analysis

**Author: OpenAI GPT-6.** This notebook analyzes JSON Lines emitted by `analyze_syndrome`: it validates the shot records, summarizes syndrome and decoder outcomes, plots logical-error rates against Hamming weight and affinity concentration, and fits a logistic regression.

The binary file stores one **effective-partner count** per detector: `(sum of off-diagonal affinities)^2 / sum of their squares`. A value near 1 means one partner dominates; larger values mean a flatter set of plausible partners. A row with no affinity has value 0. Odd-weight syndromes include a boundary detector as the final row. The syndrome mean and maximum include all stored rows.

## Setup

Set `DATA_PATH` to a `.bin.xz` (recommended) or `.bin` file, then run all cells. The binary layout is documented in `docs/affinity_binary_format.md`. This notebook needs NumPy, Matplotlib, and statsmodels. For example: `python -m pip install numpy matplotlib statsmodels`.

In [ ]:
from pathlib import Path
import lzma
import struct
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("syndromes.bin.xz")
Y_BINS = 12
MIN_SHOTS_PER_HEATMAP_BIN = 5

In [ ]:
# OpenAI GPT-6: Read the all-binary format without retaining per-detector
# arrays after their syndrome summaries have been computed.
def load_binary_syndromes(path):
    path = Path(path)
    opener = lzma.open if path.suffix == ".xz" else Path.open
    dtype = [("hamming_weight", "i4"), ("logical_error", "?"), ("mean_effective_partners", "f8"),
             ("max_effective_partners", "f8"), ("zero_rows", "i4")]
    chunk_size = 100_000
    chunks = []
    chunk = np.empty(chunk_size, dtype=dtype)
    used = 0

    def read_exact(stream, count):
        data = stream.read(count)
        if len(data) != count:
            raise ValueError("Truncated affinity binary file")
        return data

    with opener(path, "rb") as stream:
        magic, distance, rounds, p, mpi_ranks, detector_count, flags = (
            struct.unpack("<8sBBdHHB", read_exact(stream, 23)))
        if magic != b"QDAFF001" or flags & ~3 or not flags & 1:
            raise ValueError("Unsupported affinity binary header")
        if not (distance >= 3 and distance % 2 and rounds and mpi_ranks and detector_count):
            raise ValueError("Invalid affinity binary metadata")
        while True:
            prefix = stream.read(4)
            if not prefix:
                break
            if len(prefix) != 4:
                raise ValueError("Truncated affinity binary record")
            weight, boundary, logical_error = struct.unpack("<HBB", prefix)
            if boundary != weight % 2 or logical_error not in (0, 1):
                raise ValueError("Invalid boundary or logical-error flag")
            count = weight + boundary
            detector_ids = np.frombuffer(read_exact(stream, 2 * count), dtype="<u2")
            physical_ids = detector_ids[:weight]
            if (np.any(physical_ids >= detector_count) or
                    np.any(np.diff(physical_ids.astype(np.int64)) <= 0) or
                    (boundary and detector_ids[-1] != detector_count)):
                raise ValueError("Invalid detector IDs in binary record")
            effective = np.frombuffer(read_exact(stream, 8 * count), dtype="<f8")
            if not np.all(np.isfinite(effective)) or np.any(effective < 0):
                raise ValueError("Invalid effective-partner count")
            if used == chunk_size:
                chunks.append(chunk)
                chunk = np.empty(chunk_size, dtype=dtype)
                used = 0
            chunk[used] = (weight, bool(logical_error),
                           float(effective.mean()) if count else 0.0,
                           float(effective.max()) if count else 0.0,
                           int(np.count_nonzero(effective == 0)))
            used += 1

    chunks.append(chunk[:used])
    data = np.concatenate(chunks) if len(chunks) > 1 else chunks[0]
    metadata = dict(schema=3, distance=distance, rounds=rounds,
                    physical_error_rate=p, mpi_ranks=mpi_ranks,
                    detector_count=detector_count,
                    include_opposite_basis_detectors=bool(flags & 2))
    return metadata, {"completed_shots": len(data)}, data

metadata, summary, data = load_binary_syndromes(DATA_PATH)

In [ ]:
# Overview: sample size, decoder error rate, syndrome sizes, affinity
# concentration, and the prevalence of detectors with no connected partner.
shots = len(data)
errors = int(data["logical_error"].sum())
weights = data["hamming_weight"]
mean_eff = data["mean_effective_partners"]
max_eff = data["max_effective_partners"]
zero_rows = data["zero_rows"]
print(f"File: {DATA_PATH}")
print(f"Code: distance {metadata['distance']}, rounds {metadata['rounds']}, "
      f"SI1000 p={metadata['physical_error_rate']}, "
      f"opposite-basis detectors={metadata['include_opposite_basis_detectors']}")
print(f"Shots: {shots:,}  |  MPI ranks: {metadata['mpi_ranks']}  |  seed: {metadata.get('seed', 'not stored')}")
print(f"PyMatching logical errors: {errors:,} / {shots:,} = {100 * errors / shots:.4f}%")
print(f"Hamming weight: median={np.median(weights):.0f}, "
      f"mean={weights.mean():.2f}, range={weights.min()}–{weights.max()}")
print(f"Mean effective partners: median={np.median(mean_eff):.3f}, "
      f"10th/90th percentile={np.percentile(mean_eff, [10, 90])}")
print(f"Maximum effective partners: median={np.median(max_eff):.3f}, "
      f"10th/90th percentile={np.percentile(max_eff, [10, 90])}")
print(f"Syndromes with fewer than two physical detectors: {(weights < 2).sum():,}")
if metadata["schema"] in (2, 3):
    print(f"Odd-weight syndromes augmented with boundary: {(weights % 2 == 1).sum():,}")
print(f"Syndromes with at least one zero-affinity row: {(zero_rows > 0).sum():,}")
print(f"Total zero-affinity rows: {zero_rows.sum():,}")
print("Zero-affinity rows are encoded as 0 effective partners in the plots and model.")
print("Logical-error rate by Hamming weight (first 20 populated weights):")
for weight in np.unique(weights)[:20]:
    group = weights == weight
    count = int(group.sum())
    failures = int(data["logical_error"][group].sum())
    print(f"  HW {weight:3d}: {failures:5d} / {count:7d} = {100 * failures / count:7.3f}%")
if len(np.unique(weights)) > 20:
    print(f"  ... {len(np.unique(weights)) - 20} additional populated weights")

In [ ]:
# Plot exact integer Hamming-weight columns and continuous effective-partner
# bins. Each pixel is failures / shots in that bin; sparse bins are masked.
def error_rate_heatmap(data, value_field, title, y_bins=Y_BINS,
                       min_shots=MIN_SHOTS_PER_HEATMAP_BIN):
    weights = data["hamming_weight"].astype(int)
    values = data[value_field]
    failures = data["logical_error"].astype(float)
    min_weight, max_weight = int(weights.min()), int(weights.max())
    x_edges = np.arange(min_weight - 0.5, max_weight + 1.5)
    if np.isclose(values.min(), values.max()):
        y_edges = np.linspace(max(0.0, values.min() - 0.5),
                              values.max() + 0.5, 2)
    else:
        y_edges = np.linspace(values.min(), values.max(), y_bins + 1)
    counts, _, _ = np.histogram2d(weights, values, bins=(x_edges, y_edges))
    errors, _, _ = np.histogram2d(weights, values, bins=(x_edges, y_edges),
                                  weights=failures)
    rate = np.divide(100 * errors, counts, out=np.full_like(errors, np.nan),
                     where=counts >= min_shots)
    rate = np.ma.masked_invalid(rate.T)
    from matplotlib.colors import LogNorm
    fig, axes = plt.subplots(
        2, 1, figsize=(max(8, min(22, (max_weight - min_weight + 1) * 0.45)), 9),
        sharex=True, sharey=True, constrained_layout=True)
    error_ax, count_ax = axes
    error_mesh = error_ax.pcolormesh(
        x_edges, y_edges, rate, cmap="viridis", vmin=0, vmax=100, shading="flat")
    fig.colorbar(error_mesh, ax=error_ax, label="PyMatching logical-error rate (%)")
    count_image = np.ma.masked_where(counts.T == 0, counts.T)
    count_mesh = count_ax.pcolormesh(
        x_edges, y_edges, count_image, cmap="Blues",
        norm=LogNorm(vmin=1, vmax=max(1, int(counts.max()))), shading="flat")
    fig.colorbar(count_mesh, ax=count_ax, label="Shots per bin (log scale)")
    error_ax.set_title(f"{title}: logical-error rate (at least {min_shots} shots per visible bin)")
    count_ax.set_title("Shot counts; white bins have no shots")
    for ax in axes:
        ax.set_ylabel(title)
    count_ax.set_xlabel("Syndrome Hamming weight")
    count_ax.set_xticks(np.arange(
        min_weight, max_weight + 1,
        max(1, (max_weight - min_weight + 1) // 20)))
    plt.show()
    print(f"Visible error-rate bins: {int((counts >= min_shots).sum())}/{counts.size}; "
          f"shots in visible bins: {int(counts[counts >= min_shots].sum()):,}/{len(data):,}")


In [ ]:
error_rate_heatmap(data, "mean_effective_partners",
                   "Mean effective partners across fired detectors")

In [ ]:
error_rate_heatmap(data, "max_effective_partners",
                   "Maximum effective partners across fired detectors")

## Multivariate logistic regression

The outcome is the PyMatching logical-error indicator. Predictors are Hamming weight and the syndrome mean and maximum effective-partner values; every shot is included. Zero-affinity rows contribute the 0 sentinel defined above. The model reports **McFadden's pseudo R²**, because ordinary least-squares R² is not defined for logistic regression. Coefficients and odds ratios are per one-unit increase in each unscaled predictor. This is an association model; class imbalance and correlation among predictors can make estimates unstable. The cell detects one-class and fit failures.

In [ ]:
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import PerfectSeparationError

def roc_auc_from_scores(labels, scores):
    # Mann–Whitney AUC, with midranks for tied scores.
    labels = np.asarray(labels, dtype=bool)
    scores = np.asarray(scores, dtype=float)
    positives = int(labels.sum())
    negatives = len(labels) - positives
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(scores, kind="stable")
    sorted_scores = scores[order]
    ranks = np.empty(len(scores), dtype=float)
    start = 0
    while start < len(scores):
        end = start + 1
        while end < len(scores) and sorted_scores[end] == sorted_scores[start]:
            end += 1
        ranks[order[start:end]] = ((start + 1) + end) / 2.0
        start = end
    return (ranks[labels].sum() - positives * (positives + 1) / 2) / (positives * negatives)

features = np.column_stack([
    data["hamming_weight"],
    data["mean_effective_partners"],
    data["max_effective_partners"],
])
names = ["hamming_weight", "mean_effective_partners", "max_effective_partners"]
y = data["logical_error"].astype(int)
X = sm.add_constant(features, has_constant="add")
if y.min() == y.max():
    print(f"Cannot fit logistic regression: all {len(y):,} outcomes are {y[0]}.")
elif np.linalg.matrix_rank(X) < X.shape[1]:
    print("Cannot fit all requested predictors: design matrix is rank-deficient.")
else:
    try:
        model = sm.Logit(y, X)
        fit = model.fit(disp=False, maxiter=200)
        print(f"Observations: {fit.nobs:,.0f}; errors: {int(y.sum()):,}; "
              f"non-errors: {int((1-y).sum()):,}")
        print(f"Converged: {fit.mle_retvals.get('converged', False)}")
        print(f"Log-likelihood: {fit.llf:.6g}; null log-likelihood: {fit.llnull:.6g}")
        print(f"McFadden pseudo R²: {fit.prsquared:.6g}")
        print(f"Likelihood-ratio statistic: {fit.llr:.6g}; p-value: {fit.llr_pvalue:.6g}")
        print(f"AIC: {fit.aic:.6g}; BIC: {fit.bic:.6g}")
        predictions = fit.predict(X)
        print(f"In-sample ROC-AUC: {roc_auc_from_scores(y, predictions):.6g}")
        print("ROC-AUC is descriptive here; an independent test set is needed for prediction claims.")
        print(fit.summary(xname=["intercept"] + names))
        intervals = fit.conf_int()
        print()
        print("Odds ratios per unit (95% confidence intervals):")
        for i, name in enumerate(["intercept"] + names):
            print(f"  {name:30s} {np.exp(fit.params[i]):.6g} "
                  f"[{np.exp(intervals[i, 0]):.6g}, {np.exp(intervals[i, 1]):.6g}]")
    except (np.linalg.LinAlgError, ValueError, PerfectSeparationError) as exc:
        print(f"Logistic regression could not be fitted reliably: {exc}")